# Customer Review Sentiment Classification

**Author:** Tajamul Khan  
**Task:** Binary text classification  
**Primary metric:** Macro F1

## Project introduction

Classify positive and negative customer-review sentiment from review text. The original notebook's deprecated CSV loading and row-append workflow are replaced with a compact TF-IDF pipeline.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 2. Load and label reviews

Place Reviews.csv beside this notebook. Ratings 1-2 are negative, 4-5 are positive, and neutral three-star rows are excluded.

In [ ]:
DATA_FILE = 'Reviews.csv'
PROJECT_FOLDER = 'Customer Satisfaction Analysis using Classification Algorithms'
DATA_SOURCE = 'The original notebook expects the Amazon Fine Food Reviews file named Reviews.csv; the source link was not recorded in the repository.'

def resolve_data_path(filename):
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "Supervised Learning Projects" / PROJECT_FOLDER / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{filename} was not found. Place it beside the notebook. Dataset information: {DATA_SOURCE}"
    )

DATA_PATH = resolve_data_path(DATA_FILE)
raw = pd.read_csv(DATA_PATH, sep=',', low_memory=False)
print(f"Loaded {DATA_PATH.name}: {raw.shape[0]:,} rows × {raw.shape[1]} columns")

In [ ]:
required = {"Text", "Score"}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f"Required columns are missing: {sorted(missing)}")

data = raw.loc[raw["Score"].isin([1, 2, 4, 5]), ["Text", "Score"]].copy()
data["Text"] = data["Text"].fillna("").astype(str).str.strip()
data = data[data["Text"].ne("")].drop_duplicates(subset=["Text"]).reset_index(drop=True)
data["sentiment"] = data["Score"].ge(4).map({True: "positive", False: "negative"})
display(data.head())
display(data["sentiment"].value_counts().to_frame("count"))

In [ ]:
sns.countplot(data=data, x="sentiment", color="#2563eb")
plt.title("Sentiment distribution")
plt.show()
data["review_length"] = data["Text"].str.split().str.len()
display(data.groupby("sentiment")["review_length"].describe().round(1))

## 3. Stratified split and leakage-safe text pipelines

TF-IDF learns its vocabulary only from each training fold.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data["Text"], data["sentiment"], test_size=0.20,
    stratify=data["sentiment"], random_state=RANDOM_STATE,
)
models = {
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.5),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, model in models.items():
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3, max_df=0.95, sublinear_tf=True)),
        ("model", model),
    ])
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1, error_score="raise")
    rows.append({"Model": name, "CV macro F1": scores["test_score"].mean(), "CV std": scores["test_score"].std()})
comparison = pd.DataFrame(rows).sort_values("CV macro F1", ascending=False)
display(comparison.round(4))

best_name = comparison.iloc[0]["Model"]
best_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("model", models[best_name]),
])
best_pipeline.fit(X_train, y_train)

## 4. Holdout evaluation

In [ ]:
predictions = best_pipeline.predict(X_test)
print(f"Selected model: {best_name}")
print(f"Holdout macro F1: {f1_score(y_test, predictions, average='macro'):.4f}")
display(pd.DataFrame(classification_report(y_test, predictions, output_dict=True, zero_division=0)).T.round(4))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, cmap="Blues")
plt.title("Review sentiment confusion matrix")
plt.show()

## 5. Interpretation and next steps

Inspect class-specific precision and recall before using predictions operationally. Review text can contain personal information, sarcasm, spelling variation and domain drift; production use requires privacy review and monitoring. All reported values are calculated at execution time.